In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import torch
import torch.nn as nn
from torchvision import models, transforms

WEIGHTS_URL = 'https://storage.yandexcloud.net/idp-web/second/cv/efficientnet_b0.pt'
WEIGHTS_PATH = Path('/content/efficientnet_b0.pt')

CLASSES = ['apc', 'armored_vehicle', 'gun', 'ifv', 'infantry', 'mlrs', 'spg', 'tank', 'uav']
NUM_CLASSES = len(CLASSES)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)

model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()
print(f'Модель загружена  device: {DEVICE}')

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

src = input('URL изображения: ')

img_path = Path('/content/predict_input.jpg')
urlretrieve(src, img_path)
img_name = src.split('/')[-1].split('?')[0]

img = Image.open(img_path).convert('RGB')

with torch.no_grad():
    probs = torch.softmax(model(tf(img).unsqueeze(0).to(DEVICE)), dim=1)[0]

top3  = probs.topk(3)
label = CLASSES[top3.indices[0].item()]
conf  = top3.values[0].item() * 100

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img)
ax.axis('off')
ax.set_title(f'{img_name}\n{label}  {conf:.1f}%', fontsize=14, fontweight='bold', pad=12)

for rank, (idx, val) in enumerate(zip(top3.indices, top3.values)):
    ax.text(0.02, 0.97 - rank * 0.06,
            f'{CLASSES[idx.item()]}: {val.item()*100:.1f}%',
            transform=ax.transAxes, fontsize=11,
            color='white', verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))

plt.tight_layout()
plt.show()